In [5]:
import numpy as np
import pandas as pd
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator as rfg

N = 2048
GENS = {
    "ECFP4": rfg.GetMorganGenerator(radius=2, fpSize=N),
    "FCFP4": rfg.GetMorganGenerator(radius=2, fpSize=N,
                                    atomInvariantsGenerator=rfg.GetMorganFeatureAtomInvGen()),
    "RDK":   rfg.GetRDKitFPGenerator(fpSize=N),
}

def fingerprint(mol, fp_type):
    if fp_type == "MACCS":
        return np.array(list(MACCSkeys.GenMACCSKeys(mol)), dtype=np.uint8)   # 167 bits
    return GENS[fp_type].GetFingerprintAsNumPy(mol).astype(np.uint8)

def main(input_file, output_folder, smiles_col="Canonical_SMILES"):
    out = Path(output_folder)
    out.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(input_file)
    if smiles_col not in df.columns:
        print(f"Column '{smiles_col}' not found in {input_file}. Skipping.")
        return

    mols = df[smiles_col].map(lambda s: Chem.MolFromSmiles(s) if isinstance(s, str) else None)
    bad = mols.isna()
    print(f"{input_file}: {bad.sum()} invalid SMILES removed")
    df, mols = df[~bad].reset_index(drop=True), mols[~bad].tolist()

    for fp_type in ["RDK", "MACCS", "ECFP4", "FCFP4"]:
            X = np.vstack([fingerprint(m, fp_type) for m in mols])
            fp_df = pd.DataFrame(X, columns=[f"{fp_type}_{i}" for i in range(X.shape[1])])
            path = out / f"{Path(input_file).stem}_{fp_type}_fingerprints.csv"
            pd.concat([df, fp_df], axis=1).to_csv(path, index=False)
            print(f"  Saved {path.name} {X.shape}")

main("../combined_labeled_data.csv", "Fingerprints")   # use your own folder names


../combined_labeled_data.csv: 0 invalid SMILES removed
  Saved combined_labeled_data_RDK_fingerprints.csv (1885, 2048)
  Saved combined_labeled_data_MACCS_fingerprints.csv (1885, 167)
  Saved combined_labeled_data_ECFP4_fingerprints.csv (1885, 2048)
  Saved combined_labeled_data_FCFP4_fingerprints.csv (1885, 2048)
